In [1]:
import os
import sys
import yaml

sys.path.append('/home/lishengping/projects/maxtext/MaxText')
os.environ['HARDWARE'] = 'tpu'

import pyconfig
from layers import models
import max_utils
import jax
import orbax
import jax.numpy as jnp
from jax.sharding import Mesh
from flax.traverse_util import flatten_dict, unflatten_dict
from flax import linen as nn

# 配置文件需要更改的几个地方：
# query_chunk_size = None # 如果传了这个参数，forward需要是query_chunk_size的整数倍
# attention = 'dot_attention_chunk'
# exp_class set your model class
# per_device_batch_size = 1 # 可以根据测试的batch size定，设小一点主要是为了节省显存
# max_target_length = 256 # 可以根据测试的长度定，设小一点主要是为了节省显存
# zero_loss = True
# run_name = 'test' # 因为base.yml默认为空，必须写一个
# scan_layers = False # 因为转模型的时候转的是scan_layers=False
run_name = 'test'
os.makedirs(run_name, exist_ok=True) # 因为如果不存在会报错
config_name = '/home/lishengping/projects/maxtext/MaxText/configs/base.yml'
argv = [None, config_name]
config = pyconfig.initialize(argv)

2025-04-28 03:46:39.441359: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745811999.454299  182878 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745811999.458165  182878 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745811999.469258  182878 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745811999.469272  182878 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745811999.469274  182878 computation_placer.cc:177] computation placer alr

Updating keys from env and command line: []
Running Model: default
Updating keys from model: []
Attempting to initialize the jax distributed system...
Jax distributed system initialized!


Updated exp model vars:
[EXP] adam_b1: 0.9
[EXP] adam_b2: 0.95
[EXP] adam_eps: 1e-08
[EXP] adam_weight_decay: 0.1
[EXP] async_checkpointing: True
[EXP] base_emb_dim: 2048
[EXP] base_mlp_dim: 11008
[EXP] base_num_decoder_layers: 36
[EXP] base_num_kv_heads: 2
[EXP] base_num_query_heads: 16
[EXP] checkpoint_period: 250
[EXP] cosine_learning_rate_final_fraction: 0.1
[EXP] data_shuffle_seed: 9876
[EXP] dataset_type: pretrain_4k
[EXP] decoder_block: fusion
[EXP] enable_checkpointing: True
[EXP] enable_goodput_recording: False
[EXP] enable_single_replica_ckpt_restoring: False
[EXP] eval_interval: 13500
[EXP] eval_loop_num_batches: 162
[EXP] eval_per_device_batch_size: 128.0
[EXP] eval_shuffle_buffer_size: None
[EXP] head_dim: 128
[EXP] init_weights_seed: 9876
[EXP] insert_moe_indexes: []
[EXP] iter_file_num

In [2]:
config.attention, config.per_device_batch_size, config.max_target_length, config.zero_loss, config.exp_class
assert  config.query_chunk_size is None
assert not config.scan_layers

In [3]:
import json
import os
import sys
import asyncio
import argparse
from collections import defaultdict
import time

# os.environ["JAX_PLATFORMS"] = "cpu"

import torch
import numpy as np
import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64


def decode_base64(encoded_str):
    decoded_bytes = base64.b64decode(encoded_str)
    decoded_str = decoded_bytes.decode('utf-8')
    return decoded_str

def encode_base64(decoded_str):
    # decoded_str = "opt_state.mu.params.token_embedder.embedding"
    encoded_string = base64.b64encode(decoded_str.encode('utf-8')).decode('utf-8')
    return encoded_string

# 新保存一个jax版本的模型，从中读取模型的keys，之后也会基于这个模型对参数进行转换
_sharding_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/test/checkpoints/0/items/_sharding'
_sharding_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_3B_torch2jax_0425/checkpoints/0/items/_sharding'
_sharding_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_3B_torch2jax_0428/checkpoints/0/items/_sharding'

_sharding_path = epath.Path(_sharding_path)
with _sharding_path.open('r') as f:
    _sharding = json.load(f)

model_shardings = {}
for k, v in _sharding.items():
    base_k = decode_base64(k)
    base_k_split = tuple(base_k.split('.'))
    if 'opt_state' in base_k or 'step' in base_k: continue
    # print(base_k_split)
    model_shardings[base_k_split] = v

# load model
# 基于读取的jax keys，构造shape dtype 文件，便于load jax model

# 0.5B
vocab_size = 151936
base_emb_dim = 896
head_dim = 64
base_num_query_heads = 14
base_num_kv_heads = 2
base_mlp_dim = 4864
base_num_decoder_layers = 24

# 3B
vocab_size = 151936
base_emb_dim = 2048
head_dim = 128
base_num_query_heads = 16
base_num_kv_heads = 2
base_mlp_dim = 11008
base_num_decoder_layers = 36


mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
axis = [1] * len(mesh_axes)
axis[2] = 4 # tpu的chips数， v5p-8 is 4
devices = np.asarray(jax.devices()).reshape(axis)
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = np.float32 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
for k, v in model_shardings.items():
    jointk = '.'.join(k)
    if 'embedding' in jointk:
        shape = (vocab_size, base_emb_dim)
    elif 'scale' in jointk:
        shape = (base_emb_dim, )
    elif 'mlp.wi_' in jointk:
        shape = (base_emb_dim, base_mlp_dim)
    elif 'mlp.wo.' in jointk:
        shape = (base_mlp_dim, base_emb_dim)
    elif 'query.kernel' in jointk:
        shape = (base_emb_dim, base_num_query_heads, head_dim)
    elif 'query.bias' in jointk:
        shape = (base_num_query_heads, head_dim, )
    elif 'key.kernel' in jointk or 'value.kernel' in jointk:
        shape = (base_emb_dim, base_num_kv_heads, head_dim)
    elif 'key.bias' in jointk  or 'value.bias' in jointk:
        shape = (base_num_kv_heads, head_dim, )
    elif 'out.kernel' in jointk:
        shape = (base_num_query_heads, head_dim, base_emb_dim)
    else:
        print(f'Unmatched params name: {jointk}')
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)           
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)

jax_model_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/test/checkpoints/250/items'
jax_model_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_3B_torch2jax_0425/checkpoints/0/items'
jax_model_path = 'gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/qwen2.5_3B_torch2jax_0428/checkpoints/0/items'
ckpt = epath.Path(jax_model_path)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)

restored = ckptr.restore(
  ckpt, item=abstract_unboxed_params, transforms={}, restore_args=restore_args
)

('params', 'params', 'token_embedder', 'embedding') (151936, 2048)
('params', 'params', 'decoder', 'decoder_norm', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_0', 'kernel') (2048, 11008)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wi_1', 'kernel') (2048, 11008)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'mlp', 'wo', 'kernel') (11008, 2048)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'post_self_attention_layer_norm', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'pre_self_attention_layer_norm', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'out', 'kernel') (16, 128, 2048)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'bias') (2, 128)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'key', 'kernel') (2048, 2, 128)
('params', 'params', 'decoder', 'layers_4', 'sub_0', 'self_attention', 'query', 'bias

I0428 03:46:51.302211  204620 google_auth_provider.cc:181] Running on GCE, using service account 626151558586-compute@developer.gserviceaccount.com


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# model_name = "Qwen/Qwen2.5-0.5B"  # 替换为你要下载的模型
cache_dir = "/home/lishengping/3B"   # 你希望保存模型的本地路径
model = AutoModelForCausalLM.from_pretrained(
    cache_dir,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(cache_dir)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
def model_init(model, config, key):
  input_shape = (config.global_batch_size_to_load, config.max_target_length)
  params = model.init(
      {"params": key, "dropout": key, "aqt": key},
      jnp.ones(input_shape, dtype=jnp.int32),
      jnp.ones(input_shape, dtype=jnp.int32),
  )
  return params

quant = None
devices_array = max_utils.create_device_mesh(config)
mesh = Mesh(devices_array, config.mesh_axes)
Transformer = models.Transformer
jax_model = Transformer(config, mesh, quant=quant)

is_train = False
rng1, aqt_rng = jax.random.split(jax.random.key(9876))
# 这个model_init是模型真实的参数，可以和加载的参数去对比，看有什么区别
# init_params = model_init(jax_model, config, rng1)

inp = '''<|im_start|>一个则非常实用，而且轻便，扛在肩上没有负担。\n片断五\n诵读：阿宽 如喜 朗月 张凤霞\n薛如刚 云在飞 牧歌\n彼得大叔来了\n和平到来后的那个温暖的夏天,塔拉突然间失去了往昔的宁静。接下来的几个月里,一队队士兵拖着艰难的脚步,吃力地翻过那座红色的山丘来到塔拉,在门前台阶的阴凉处歇息,衣衫槛楼、胡子拉碴、步履蹒跚、饥肠辘辘,盼望得到食物,想要投宿一夜。他们是正在返家的南军士兵。火车将约翰斯顿残部的士兵从北卡罗来纳州运送到亚特兰大,将他们扔在那儿,从此,这些士兵们就开始了徒步跋涉,走上回家的路。可是,他们的家或许已经不复存在,家里的人也许是死的死,散的散了。\n他们对眼前的困难不屑一顾,一切都结束了。回家!回家!这是士兵们脑子里的惟一念头。他们正在往家里赶,这是他们惟一的支撑。他们打仗已经尽了全力,结果被打败了,如今他们很愿意安定下来,在他们反对过的旗帜下安居乐业。\n斯佳丽和玫兰尼向每一个士兵急切地打听阿希礼的消息。但是谁也没听说过他的消息,而且他们也不愿谈起失踪人员的事情。他们自己活着就足够了,他们不关心也不愿去想那数以千计躺在无名坟慕里永远也回不了家的人。\n每次失望后,家里人都努力安慰玫兰妮.让她保持信心。阿希礼肯定没有死在俘虏营中。否则北佬的牧师会写信通知他们的。他准是在回家的路上,不过他的俘虏营离得那么远。天啊,这么远的路火车都得走好几天,要是阿希礼和这些人一样步行的话......\n六月的一个下午,塔拉的人都聚集在后门廊,热切地看着波克切开这年第一个半生不熟的西瓜,他们听到屋前的碎石路上传来了马蹄声。普莉西不情愿地朝前门走去,其他人则在她身后激烈地讨论,要是来者是个当兵的,他们是该把西瓜藏起来呢还是留下来晚饭时招待客人。\n波克抱着小西瓜站在那里,不知所措,这时他们听到普莉西的喊声。\n“老天爷啊!斯佳丽小姐!玫荔小姐!快来!”\n“是谁啊?”斯佳丽一边喊,一边从台阶上跳了起来,穿过厅堂朝外冲去,玫荔紧跟在她的身后,其他人也都跟着往外跑。\n“是阿希礼!”斯佳丽心想。“哦,可能.....\n“是彼得大叔!佩蒂小姐家的彼得大叔!”大家都跑到了前门廊,看见那个高个子、灰白头发的老管家正从一匹绑着被子当马鞍、长着一条老鼠尾巴的老马背上往下爬。他那张宽宽的黑脸上总是摆出一副很有尊严的表情,现在看见了老朋友虽然非常高兴却又不想放弃尊严,结果是他的眉头紧锁,嘴巴却咧开'''
inp = '''在这样雨雪交加的日子里，如果没有什么紧要事，人们宁愿一整天足不出户。因此，县城的大街小巷倒也比平时少了许多嘈杂。街巷背阴的地方。冬天残留的积雪和冰溜子正在雨点的敲击下蚀化，石板街上到处都漫流着肮脏的污水。风依然是寒冷的。空荡荡的街道上，有时会偶尔走过来一个乡下人，破毡帽护着脑门，胳膊上挽一筐子土豆或萝卜，有气无力地呼唤着买主。唉，城市在这样的日子里完全丧失了生气，变得没有一点可爱之处了。 
只有在半山腰县立高中的大院坝里，此刻却自有一番热闹景象。午饭铃声刚刚响过，从一排排高低错落的石窑洞里，就跑出来了一群一伙的男男女女。他们把碗筷敲得震天价响，踏泥带水、叫叫嚷嚷地跑过院坝，向南面总务处那一排窑洞的墙根下蜂涌而去。偌大一个院子，霎时就被这纷乱的人群踩踏成了一片烂泥滩。与此同时，那些家在本城的走读生们，也正三三两两涌出东面学校的大门。他们撑着雨伞，一路说说笑笑，通过一段早年间用横石片插起的长长的下坡路，不多时便纷纷消失在城市的大街小巷中。 
在校园内的南墙根下，现在已经按班级排起了十几路纵队。各班的值日生正在忙碌地给众人分饭菜。每个人的饭菜都是昨天登记好并付了饭票的，因此程序并不复杂，现在值日生只是按饭表付给每人预订的一份。菜分甲、乙、丙三等。甲菜以土豆、白菜、粉条为主，里面有些叫人嘴馋的大肉片，每份三毛钱；乙菜其它内容和甲菜一样，只是没有肉，每份一毛五分钱。丙菜可就差远了，清水煮白萝卜——似乎只是为了掩饰这过分的清淡，才在里面象征性地漂了几点辣子油花。不过，这菜价钱倒也便宜，每份五分钱。 '''
# inp = '''飞流直下三千尺，疑是银河落九天。'''
# inp = '<|im_start|>我们回家'
inp = '李青峰'

batch_size = 1
x = tokenizer.encode(inp)
input_ids = jnp.array(x).reshape(batch_size, -1)
data = {}
data['inputs'] = input_ids[:, :-1]
data["inputs_position"] = jnp.arange(data['inputs'].shape[1]).reshape(batch_size, -1)
data["inputs_segmentation"] = jnp.ones_like(data['inputs'])
data["targets"] = input_ids[:, 1:]

Num_devices: 4, shape (1, 1, 4, 1, 1, 1, 1, 1, 1, 1)


In [ ]:
params = restored['params']['params']
jax_logits, intermediate_outputs = jax_model.apply(
      {'params': params},
      data["inputs"],
      data["inputs_position"],
      decoder_segment_ids=data["inputs_segmentation"],
      enable_dropout=config.enable_dropout if is_train else False,
      rngs={"dropout": rng1, "params": aqt_rng},
      mutable="intermediates",
  )
one_hot_targets = jax.nn.one_hot(data["targets"], config.vocab_size)
jax_loss, _ = max_utils.cross_entropy_with_logits(jax_logits, one_hot_targets, 0.0)
print(f'jax mean loss: {jax_loss.mean()}')

logits_via_embedding11: True
jax mean loss: 6.179162979125977


In [ ]:
torch_input_ids = torch.tensor(x).reshape(batch_size, -1)
model.eval()
# model = model.half()
model = model.to(torch.bfloat16)
with torch.no_grad():
    torch_outputs = model.forward(torch_input_ids, labels=torch_input_ids.long())
print(f'torch mean loss: {torch_outputs.loss}')

torch mean loss: 6.182320594787598


In [ ]:
# # train
# BUCKET_ZONE=europe-west4
# ZONE=$BUCKET_ZONE-b
# TPU_TYPE=v5p-8
# TPU_NAME=llm-jax-$TPU_TYPE-10
# BASE_OUTPUT_DIR=gs://newproject-1-llm_base_models_$BUCKET_ZONE/v5p_256/7B/
# RUN_NAME=qwen2.5_3B_torch2jax_0428
# DATASET_PATH=gs://newproject-1-jax_llm_data_europe-west4/xiaomeng/v3.5/tfids1210
# WORK_DIR=/home/lishengping/projects/maxtext
# pip3 install -U jax[tpu]==0.5.1 jaxlib[tpu]==0.5.1 orbax-checkpoint==0.11.1 aqtp pathwaysutils flax omegaconf google-cloud-monitoring  cloud-accelerator-diagnostics cloud-tpu-diagnostics ml_goodput_measurement tensorflow-text
# python MaxText/train.py MaxText/configs/base.yml  base_output_directory=$BASE_OUTPUT_DIR run_name=$RUN_NAME  dataset_path=$DATASET_PATH exp_class='Qwen2p5_3B' enable_checkpointing=True dataset_type='pretrain_4k' query_chunk_size=512 attention='dot_product_chunk' scan_layers=True debug=True record_internal_nn_metrics=0 eval_interval=13500 per_device_batch_size=4 eval_per_device_batch_size=16.0 max_target_length=1024 num_experts=0 load_parameters_path='gs://newproject-1-llm_base_models_europe-west4/v5p_256/7B/RUN_NAME=qwen2.5_3B_torch2jax_0428/checkpoints/0' | tee train.log